In [2]:
from langchain_core.documents import Document

sample_doc = Document(
    page_content = '직전 학기 평점 3.5 이상인 학생에게 성적 우수 장학금을 지급합니다.',
    metadata={
        "source": "2026_학사규정.pdf",
        "page": 12,
        "category": "장학",
        "year": 2026
    }
)

print("===[본문 데이터]===")
print(sample_doc.page_content)
print("\n===[부착된 메타데이터]===")
print(sample_doc.metadata)

===[본문 데이터]===
직전 학기 평점 3.5 이상인 학생에게 성적 우수 장학금을 지급합니다.

===[부착된 메타데이터]===
{'source': '2026_학사규정.pdf', 'page': 12, 'category': '장학', 'year': 2026}


In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

# 원본 문서 내용과 파일 메타데이터를 정의합니다.
raw_text = (
    "제1장 총칙: 본 규정은 컴퓨터공학과의 실습실 이용 규정을 명시합니다.\n\n"
    "제2장 이용 시간: 평일은 오전 9시부터 오후 10시까지 개방합니다. 주말은 미개방합니다."
)
file_info = {"source": "cs_lab_policy.pdf", "department": "컴퓨터공학과", "version": "v1.2"}

original_document = Document(page_content=raw_text, metadata=file_info)

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=40,
    chunk_overlap=0
)

chunked_docs = text_splitter.split_documents([original_document])

for i, doc in enumerate(chunked_docs):
    doc.metadata['chunk_id'] = i+1
    print(f"청크{doc.metadata['chunk_id']} 내용: {doc.page_content.strip()}")
    print(f"청크{doc.metadata['chunk_id']} 메타데이터: {doc.metadata}\n")

청크1 내용: 제1장 총칙: 본 규정은 컴퓨터공학과의 실습실 이용 규정을 명시합니다.
청크1 메타데이터: {'source': 'cs_lab_policy.pdf', 'department': '컴퓨터공학과', 'version': 'v1.2', 'chunk_id': 1}

청크2 내용: 제2장 이용 시간: 평일은 오전 9시부터 오후 10시까지 개방합니다.
청크2 메타데이터: {'source': 'cs_lab_policy.pdf', 'department': '컴퓨터공학과', 'version': 'v1.2', 'chunk_id': 2}

청크3 내용: 주말은 미개방합니다.
청크3 메타데이터: {'source': 'cs_lab_policy.pdf', 'department': '컴퓨터공학과', 'version': 'v1.2', 'chunk_id': 3}



In [4]:
# 다양한 메타데이터를 포함하는 청크 리스트를 생성합니다.
documents = [
    Document(
        page_content="컴퓨터공학과 전공자는 졸업 필수 요건으로 캡스톤 디자인을 이수해야 합니다.",
        metadata={"dept": "컴공", "year": 2024, "type": "졸업"}
    ),
    Document(
        page_content="컴퓨터공학과 전공자는 산학 프로젝트 또는 캡스톤 디자인 중 1개를 택할 수 있습니다.",
        metadata={"dept": "컴공", "year": 2026, "type": "졸업"}
    ),
    Document(
        page_content="경영학과 전공자는 전공 필수 36학점을 이수해야 졸업이 가능합니다.",
        metadata={"dept": "경영", "year": 2026, "type": "졸업"}
    )
]

target_dept = "컴공"
target_year = 2026

filtered_documents = [
    doc for doc in documents
    if doc.metadata.get('dept') == target_dept and doc.metadata.get('year') == target_year
]

print(f"=== [ 필터링 결과: {target_dept} / {target_year}년도 ] ===")

for doc in filtered_documents:
    print(f"내용: {doc.page_content}")
    print(f"속성: {doc.metadata}")

=== [ 필터링 결과: 컴공 / 2026년도 ] ===
내용: 컴퓨터공학과 전공자는 산학 프로젝트 또는 캡스톤 디자인 중 1개를 택할 수 있습니다.
속성: {'dept': '컴공', 'year': 2026, 'type': '졸업'}


In [6]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

model = init_chat_model(
    model_provider='openai',
    model="gpt-6-luna"
)

selected_doc = filtered_documents[0]

# 출처 정보를 명시할 수 있도록 프롬프트 템플릿을 설계합니다.
prompt = ChatPromptTemplate.from_messages([
    ("system", (
        "제공된 [참고 문서]에만 근거하여 명확하게 답변하세요.\n"
        "답변 끝에는 반드시 [출처: 문서종류 (연도)] 형식으로 근거를 명시하세요.\n\n"
        "[참고 문서]:\n{context}\n\n"
        "[문서 메타데이터]: 학과={dept}, 연도={year}"
    )),
    ("user", "{question}")
])

rag_chain = prompt | model | StrOutputParser()

user_query = "컴퓨터공학과 졸업을 위한 필수 요건이 무엇인가요?"
response = rag_chain.invoke({
    "context": selected_doc.page_content,
    "dept": selected_doc.metadata["dept"],
    "year": selected_doc.metadata["year"],
    "question": user_query
})

print("=== [모델 답변] ===")
print(response)

=== [모델 답변] ===
제공된 문서에 따르면 컴퓨터공학과 전공자는 **산학 프로젝트와 캡스톤 디자인 중 하나를 선택할 수 있습니다.** 다만 이것이 졸업을 위한 필수 요건인지, 또는 다른 졸업 요건이 있는지는 문서에 명시되어 있지 않습니다. [출처: 참고 문서 (2026)]
